# Recurrent Neural Networks (RNN) & Long Short-Term Memory (LSTM)

This notebook covers:
1. **RNN** — Mathematical foundations, from-scratch implementation, and PyTorch implementation
2. **LSTM** — Mathematical foundations, from-scratch implementation, and PyTorch implementation
3. **Time Complexity** analysis of each architecture

---

## Why Recurrence?

Feedforward networks treat each input independently. For **sequential data** (text, time series, audio), we need models that maintain a **hidden state** — a compressed representation of all previously seen inputs. RNNs achieve this via a recurrence relation; LSTMs extend this with gating mechanisms to handle long-range dependencies.

---
# Part 1: Recurrent Neural Network (RNN)

## 1.1 Mathematical Foundation

### Forward Pass

At each time step $$t$$, the RNN computes:

$$h_t = \tanh(W_{xh} \cdot x_t + W_{hh} \cdot h_{t-1} + b_h)$$

$$y_t = W_{hy} \cdot h_t + b_y$$

Where:
| Symbol | Shape | Description |
|--------|-------|-------------|
| $$x_t$$ | $$(d, 1)$$ | Input vector at time $$t$$ |
| $$h_t$$ | $$(n, 1)$$ | Hidden state at time $$t$$ |
| $$h_{t-1}$$ | $$(n, 1)$$ | Previous hidden state |
| $$W_{xh}$$ | $$(n, d)$$ | Input-to-hidden weight matrix |
| $$W_{hh}$$ | $$(n, n)$$ | Hidden-to-hidden (recurrent) weight matrix |
| $$W_{hy}$$ | $$(o, n)$$ | Hidden-to-output weight matrix |
| $$b_h$$ | $$(n, 1)$$ | Hidden bias |
| $$b_y$$ | $$(o, 1)$$ | Output bias |
| $$d$$ | scalar | Input dimensionality |
| $$n$$ | scalar | Hidden state dimensionality |
| $$o$$ | scalar | Output dimensionality |

### Backpropagation Through Time (BPTT)

The loss at each time step $$t$$ is $$L_t$$. Total loss: $$L = \sum_{t=1}^{T} L_t$$

Gradient of loss w.r.t. $$W_{hh}$$ requires unrolling through time:

$$\frac{\partial L}{\partial W_{hh}} = \sum_{t=1}^{T} \sum_{k=1}^{t} \frac{\partial L_t}{\partial h_t} \cdot \left(\prod_{j=k+1}^{t} \frac{\partial h_j}{\partial h_{j-1}}\right) \cdot \frac{\partial h_k}{\partial W_{hh}}$$

Where: $$\frac{\partial h_j}{\partial h_{j-1}} = \text{diag}(1 - h_j^2) \cdot W_{hh}$$

**Vanishing/Exploding Gradient Problem:**
The product $$\prod_{j=k+1}^{t} \frac{\partial h_j}{\partial h_{j-1}}$$ involves repeated multiplication of $$W_{hh}$$. If the largest singular value of $$W_{hh}$$ is:
- $$< 1$$: gradients **vanish** exponentially → cannot learn long-range dependencies
- $$> 1$$: gradients **explode** exponentially → unstable training

### Time Complexity

| Operation | Complexity | Explanation |
|-----------|-----------|-------------|
| Single step forward | $$O(n^2 + nd + no)$$ | Matrix multiplications: $$W_{hh} \cdot h$$ is $$O(n^2)$$, $$W_{xh} \cdot x$$ is $$O(nd)$$, $$W_{hy} \cdot h$$ is $$O(no)$$ |
| Full sequence forward | $$O(T(n^2 + nd + no))$$ | $$T$$ sequential steps (cannot be parallelized) |
| BPTT backward | $$O(T(n^2 + nd + no))$$ | Backprop through $$T$$ steps |
| **Total training** | $$O(T(n^2 + nd + no))$$ | Dominated by hidden-to-hidden: $$O(Tn^2)$$ when $$n > d, o$$ |

**Space Complexity:** $$O(Tn)$$ — must store all hidden states for backprop.

In [0]:
import numpy as np

class RNNFromScratch:
    """
    Vanilla RNN implemented from scratch using NumPy.
    Supports forward pass, loss computation, and full BPTT.
    """
    
    def __init__(self, input_size, hidden_size, output_size, lr=0.01):
        """
        Args:
            input_size (int): Dimensionality of input vectors (d)
            hidden_size (int): Dimensionality of hidden state (n)
            output_size (int): Dimensionality of output (o)
            lr (float): Learning rate
        """
        self.hidden_size = hidden_size
        self.lr = lr
        
        # Xavier initialization for stable gradients
        scale_xh = np.sqrt(2.0 / (input_size + hidden_size))
        scale_hh = np.sqrt(2.0 / (hidden_size + hidden_size))
        scale_hy = np.sqrt(2.0 / (hidden_size + output_size))
        
        # Weight matrices
        self.W_xh = np.random.randn(hidden_size, input_size) * scale_xh    # (n, d)
        self.W_hh = np.random.randn(hidden_size, hidden_size) * scale_hh   # (n, n)
        self.W_hy = np.random.randn(output_size, hidden_size) * scale_hy   # (o, n)
        
        # Biases
        self.b_h = np.zeros((hidden_size, 1))   # (n, 1)
        self.b_y = np.zeros((output_size, 1))   # (o, 1)
    
    def forward(self, inputs, h_prev):
        """
        Forward pass through the entire sequence.
        
        Args:
            inputs: list of input vectors, each shape (d, 1)
            h_prev: initial hidden state, shape (n, 1)
            
        Returns:
            outputs: list of output vectors (before softmax)
            hidden_states: dict of hidden states for BPTT
        """
        hidden_states = {-1: h_prev.copy()}
        raw_states = {}  # pre-activation values
        outputs = []
        
        for t, x_t in enumerate(inputs):
            # h_t = tanh(W_xh * x_t + W_hh * h_{t-1} + b_h)
            raw = self.W_xh @ x_t + self.W_hh @ hidden_states[t - 1] + self.b_h
            hidden_states[t] = np.tanh(raw)
            raw_states[t] = raw
            
            # y_t = W_hy * h_t + b_y
            y_t = self.W_hy @ hidden_states[t] + self.b_y
            outputs.append(y_t)
        
        self._cache = {
            'inputs': inputs,
            'hidden_states': hidden_states,
            'raw_states': raw_states
        }
        return outputs, hidden_states
    
    def softmax(self, x):
        """Numerically stable softmax."""
        e_x = np.exp(x - np.max(x))
        return e_x / e_x.sum(axis=0)
    
    def compute_loss(self, outputs, targets):
        """
        Cross-entropy loss over the sequence.
        
        Args:
            outputs: list of raw output vectors
            targets: list of integer class labels
        Returns:
            total loss (scalar)
        """
        loss = 0.0
        self._probs = []
        for y, t in zip(outputs, targets):
            p = self.softmax(y)
            self._probs.append(p)
            loss += -np.log(p[t, 0] + 1e-8)
        return loss
    
    def backward(self, targets):
        """
        Backpropagation Through Time (BPTT).
        Computes gradients for all parameters.
        
        Args:
            targets: list of integer class labels
        Returns:
            gradients dict
        """
        inputs = self._cache['inputs']
        hidden_states = self._cache['hidden_states']
        T = len(inputs)
        
        # Initialize gradients
        dW_xh = np.zeros_like(self.W_xh)
        dW_hh = np.zeros_like(self.W_hh)
        dW_hy = np.zeros_like(self.W_hy)
        db_h = np.zeros_like(self.b_h)
        db_y = np.zeros_like(self.b_y)
        
        dh_next = np.zeros_like(hidden_states[0])  # gradient from future
        
        for t in reversed(range(T)):
            # Gradient of loss w.r.t. output (softmax + cross-entropy)
            dy = self._probs[t].copy()
            dy[targets[t]] -= 1  # dL/dy = p - one_hot(target)
            
            # Gradients for output layer
            dW_hy += dy @ hidden_states[t].T
            db_y += dy
            
            # Gradient flowing into hidden state
            dh = self.W_hy.T @ dy + dh_next
            
            # Backprop through tanh: d/dx tanh(x) = 1 - tanh^2(x)
            dh_raw = dh * (1 - hidden_states[t] ** 2)
            
            # Gradients for hidden layer
            dW_xh += dh_raw @ inputs[t].T
            dW_hh += dh_raw @ hidden_states[t - 1].T
            db_h += dh_raw
            
            # Propagate gradient to previous time step
            dh_next = self.W_hh.T @ dh_raw
        
        # Gradient clipping to prevent exploding gradients
        for grad in [dW_xh, dW_hh, dW_hy, db_h, db_y]:
            np.clip(grad, -5, 5, out=grad)
        
        return {'dW_xh': dW_xh, 'dW_hh': dW_hh, 'dW_hy': dW_hy, 'db_h': db_h, 'db_y': db_y}
    
    def update(self, grads):
        """SGD parameter update."""
        self.W_xh -= self.lr * grads['dW_xh']
        self.W_hh -= self.lr * grads['dW_hh']
        self.W_hy -= self.lr * grads['dW_hy']
        self.b_h -= self.lr * grads['db_h']
        self.b_y -= self.lr * grads['db_y']


# --- Demo: Character-level language model ---
print("="*60)
print("RNN FROM SCRATCH - Character-level Language Model")
print("="*60)

# Tiny corpus
text = "hello world hello neural network"
chars = sorted(set(text))
char_to_idx = {c: i for i, c in enumerate(chars)}
idx_to_char = {i: c for i, c in enumerate(chars)}
vocab_size = len(chars)

print(f"\nCorpus: '{text}'")
print(f"Vocabulary size: {vocab_size}")
print(f"Characters: {chars}")

# Hyperparameters
hidden_size = 64
seq_length = 10

# Initialize model
rnn = RNNFromScratch(input_size=vocab_size, hidden_size=hidden_size, output_size=vocab_size, lr=0.01)

# Training loop
losses = []
for epoch in range(200):
    h_prev = np.zeros((hidden_size, 1))
    epoch_loss = 0
    
    for i in range(0, len(text) - seq_length, seq_length):
        # Prepare one-hot encoded inputs and targets
        inputs = []
        targets = []
        for j in range(seq_length):
            x = np.zeros((vocab_size, 1))
            x[char_to_idx[text[i + j]]] = 1
            inputs.append(x)
            targets.append(char_to_idx[text[i + j + 1]] if i + j + 1 < len(text) else char_to_idx[text[0]])
        
        # Forward
        outputs, hidden_states = rnn.forward(inputs, h_prev)
        loss = rnn.compute_loss(outputs, targets)
        epoch_loss += loss
        
        # Backward
        grads = rnn.backward(targets)
        rnn.update(grads)
        
        # Carry hidden state forward (truncated BPTT)
        h_prev = hidden_states[seq_length - 1]
    
    losses.append(epoch_loss)
    if epoch % 50 == 0:
        print(f"Epoch {epoch:3d} | Loss: {epoch_loss:.4f}")

print(f"\nFinal Loss: {losses[-1]:.4f} (started at {losses[0]:.4f})")
print(f"Loss reduction: {((losses[0] - losses[-1]) / losses[0] * 100):.1f}%")

RNN FROM SCRATCH - Character-level Language Model

Corpus: 'hello world hello neural network'
Vocabulary size: 13
Characters: [' ', 'a', 'd', 'e', 'h', 'k', 'l', 'n', 'o', 'r', 't', 'u', 'w']
Epoch   0 | Loss: 81.6389
Epoch  50 | Loss: 1.7514
Epoch 100 | Loss: 0.6500
Epoch 150 | Loss: 0.3843

Final Loss: 0.2709 (started at 81.6389)
Loss reduction: 99.7%


In [0]:
import torch
import torch.nn as nn
import torch.optim as optim

class RNNPyTorch(nn.Module):
    """
    RNN implemented using PyTorch's nn.Module.
    Demonstrates both manual RNNCell usage and nn.RNN.
    """
    
    def __init__(self, input_size, hidden_size, output_size, num_layers=1):
        super().__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        
        # PyTorch's optimized RNN layer
        # Internally computes: h_t = tanh(W_ih @ x_t + b_ih + W_hh @ h_{t-1} + b_hh)
        self.rnn = nn.RNN(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True  # Input shape: (batch, seq_len, input_size)
        )
        
        # Output projection: hidden state -> output classes
        self.fc = nn.Linear(hidden_size, output_size)
    
    def forward(self, x, h_0=None):
        """
        Args:
            x: (batch_size, seq_len, input_size)
            h_0: (num_layers, batch_size, hidden_size) or None
        Returns:
            output: (batch_size, seq_len, output_size)
            h_n: final hidden state
        """
        if h_0 is None:
            h_0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size, device=x.device)
        
        # rnn_out: (batch, seq_len, hidden_size) - all hidden states
        # h_n: (num_layers, batch, hidden_size) - final hidden state
        rnn_out, h_n = self.rnn(x, h_0)
        
        # Project all hidden states to output
        output = self.fc(rnn_out)
        return output, h_n


# --- Demo: Same character-level task with PyTorch ---
print("="*60)
print("RNN USING PYTORCH - Character-level Language Model")
print("="*60)

# Prepare data
text = "hello world hello neural network"
chars = sorted(set(text))
char_to_idx = {c: i for i, c in enumerate(chars)}
vocab_size = len(chars)
seq_length = 10
hidden_size = 64

# Encode full text
encoded = [char_to_idx[c] for c in text]

# Create training sequences
X_seqs, Y_seqs = [], []
for i in range(0, len(encoded) - seq_length):
    X_seqs.append(encoded[i:i+seq_length])
    Y_seqs.append(encoded[i+1:i+seq_length+1])

# Convert to tensors - one-hot encode inputs
X_tensor = torch.zeros(len(X_seqs), seq_length, vocab_size)
for i, seq in enumerate(X_seqs):
    for j, idx in enumerate(seq):
        X_tensor[i, j, idx] = 1.0

Y_tensor = torch.LongTensor(Y_seqs)  # (num_seqs, seq_length)

print(f"Training sequences: {X_tensor.shape[0]}")
print(f"Input shape: {X_tensor.shape} (batch, seq_len, vocab_size)")
print(f"Target shape: {Y_tensor.shape}")

# Initialize model
model = RNNPyTorch(input_size=vocab_size, hidden_size=hidden_size, output_size=vocab_size)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
print(f"\nTotal parameters: {total_params:,}")
print(f"  RNN layer: {sum(p.numel() for p in model.rnn.parameters()):,}")
print(f"  FC layer: {sum(p.numel() for p in model.fc.parameters()):,}")

# Training
losses = []
for epoch in range(200):
    optimizer.zero_grad()
    output, _ = model(X_tensor)  # (batch, seq_len, vocab_size)
    
    # Reshape for cross-entropy: (batch*seq_len, vocab_size) vs (batch*seq_len,)
    loss = criterion(output.view(-1, vocab_size), Y_tensor.view(-1))
    loss.backward()
    
    # Gradient clipping
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
    optimizer.step()
    
    losses.append(loss.item())
    if epoch % 50 == 0:
        print(f"Epoch {epoch:3d} | Loss: {loss.item():.4f}")

print(f"\nFinal Loss: {losses[-1]:.4f}")

# Generate text
model.eval()
with torch.no_grad():
    seed = "hell"
    generated = seed
    h = None
    
    for char in seed:
        x = torch.zeros(1, 1, vocab_size)
        x[0, 0, char_to_idx[char]] = 1.0
        _, h = model(x, h)
    
    # Generate 20 more characters
    x = torch.zeros(1, 1, vocab_size)
    x[0, 0, char_to_idx[seed[-1]]] = 1.0
    
    for _ in range(20):
        out, h = model(x, h)
        probs = torch.softmax(out[0, 0], dim=0)
        idx = torch.multinomial(probs, 1).item()
        generated += chars[idx]
        x = torch.zeros(1, 1, vocab_size)
        x[0, 0, idx] = 1.0
    
    print(f"\nGenerated text (seed='{seed}'): '{generated}'")

RNN USING PYTORCH - Character-level Language Model
Training sequences: 22
Input shape: torch.Size([22, 10, 13]) (batch, seq_len, vocab_size)
Target shape: torch.Size([22, 10])

Total parameters: 5,901
  RNN layer: 5,056
  FC layer: 845
Epoch   0 | Loss: 2.5649
Epoch  50 | Loss: 0.1130
Epoch 100 | Loss: 0.0998
Epoch 150 | Loss: 0.0972

Final Loss: 0.0961

Generated text (seed='hell'): 'hell neural networkd hel'


---
# Part 2: Long Short-Term Memory (LSTM)

## 2.1 Mathematical Foundation

LSTM solves the vanishing gradient problem by introducing a **cell state** $$C_t$$ (a "memory highway") controlled by three **gates**.

### Gate Equations

At each time step $$t$$:

**1. Forget Gate** — decides what to discard from cell state:
$$f_t = \sigma(W_f \cdot [h_{t-1}, x_t] + b_f)$$

**2. Input Gate** — decides what new information to store:
$$i_t = \sigma(W_i \cdot [h_{t-1}, x_t] + b_i)$$

**3. Candidate Cell State** — proposes new values:
$$\tilde{C}_t = \tanh(W_C \cdot [h_{t-1}, x_t] + b_C)$$

**4. Cell State Update** — selective forget + selective write:
$$C_t = f_t \odot C_{t-1} + i_t \odot \tilde{C}_t$$

**5. Output Gate** — decides what to expose as hidden state:
$$o_t = \sigma(W_o \cdot [h_{t-1}, x_t] + b_o)$$

**6. Hidden State** — filtered cell state:
$$h_t = o_t \odot \tanh(C_t)$$

Where:
- $$\sigma$$ is the sigmoid function: $$\sigma(x) = \frac{1}{1+e^{-x}}$$
- $$\odot$$ denotes element-wise (Hadamard) multiplication
- $$[h_{t-1}, x_t]$$ is concatenation of previous hidden state and current input

### Dimensions

| Symbol | Shape | Description |
|--------|-------|-------------|
| $$x_t$$ | $$(d, 1)$$ | Input at time $$t$$ |
| $$h_t$$ | $$(n, 1)$$ | Hidden state |
| $$C_t$$ | $$(n, 1)$$ | Cell state (long-term memory) |
| $$f_t, i_t, o_t$$ | $$(n, 1)$$ | Gate activations (values in $$[0, 1]$$) |
| $$W_f, W_i, W_C, W_o$$ | $$(n, n+d)$$ | Weight matrices (operate on concatenated $$[h, x]$$) |
| $$b_f, b_i, b_C, b_o$$ | $$(n, 1)$$ | Bias vectors |

### Why LSTM Solves Vanishing Gradients

The gradient of loss w.r.t. $$C_k$$ (cell state at step $$k$$) through time:

$$\frac{\partial C_t}{\partial C_k} = \prod_{j=k+1}^{t} f_j$$

Since $$f_j \in (0, 1)$$ and is **learned** (not a fixed weight matrix), the network can:
- Set $$f_j \approx 1$$ to **preserve** gradients over long distances ("remember")
- Set $$f_j \approx 0$$ to **reset** the cell ("forget")

This is fundamentally different from vanilla RNN where the gradient product involves $$W_{hh}$$ repeatedly.

### Backpropagation in LSTM

The gradient flows through TWO paths:
1. **Cell state path**: $$C_t \to C_{t-1}$$ (multiplicative, gated by $$f_t$$)
2. **Hidden state path**: $$h_t \to h_{t-1}$$ (through gate computations)

This dual-path structure ensures gradients can flow unimpeded when needed.

### Time Complexity

| Operation | Complexity | Explanation |
|-----------|-----------|-------------|
| Single step forward | $$O(n(n+d))$$ | 4 gate computations, each is a matrix-vector multiply on concatenated $$[h, x]$$ of size $$(n+d)$$ |
| Full sequence forward | $$O(4T \cdot n(n+d))$$ | 4 gates $$\times$$ $$T$$ steps |
| BPTT backward | $$O(4T \cdot n(n+d))$$ | Backprop through all gates at all time steps |
| **Total training** | $$O(T \cdot n(n+d))$$ | Simplified: $$\approx O(Tn^2)$$ when $$n > d$$ |

**Comparison with RNN:** LSTM has \~4x the computation per step (4 gates vs 1 state update), but the constant factor is worth it for the ability to learn long-range dependencies.

**Space Complexity:** $$O(Tn)$$ — must store $$h_t$$, $$C_t$$, and all gate values for each step.

In [0]:
import numpy as np

class LSTMFromScratch:
    """
    LSTM implemented from scratch using NumPy.
    Full forward pass with gating mechanism and BPTT.
    """
    
    def __init__(self, input_size, hidden_size, output_size, lr=0.01):
        """
        Args:
            input_size (int): Dimensionality of input (d)
            hidden_size (int): Hidden/cell state size (n)
            output_size (int): Output dimensionality (o)
            lr (float): Learning rate
        """
        self.hidden_size = hidden_size
        self.input_size = input_size
        self.output_size = output_size
        self.lr = lr
        
        concat_size = hidden_size + input_size  # n + d
        
        # Xavier initialization
        scale = np.sqrt(2.0 / (concat_size + hidden_size))
        
        # Forget gate parameters
        self.W_f = np.random.randn(hidden_size, concat_size) * scale
        self.b_f = np.ones((hidden_size, 1))  # Initialize to 1 (remember by default)
        
        # Input gate parameters
        self.W_i = np.random.randn(hidden_size, concat_size) * scale
        self.b_i = np.zeros((hidden_size, 1))
        
        # Candidate cell state parameters
        self.W_c = np.random.randn(hidden_size, concat_size) * scale
        self.b_c = np.zeros((hidden_size, 1))
        
        # Output gate parameters
        self.W_o = np.random.randn(hidden_size, concat_size) * scale
        self.b_o = np.zeros((hidden_size, 1))
        
        # Output projection
        self.W_y = np.random.randn(output_size, hidden_size) * np.sqrt(2.0 / (hidden_size + output_size))
        self.b_y = np.zeros((output_size, 1))
    
    def sigmoid(self, x):
        """Numerically stable sigmoid."""
        return np.where(x >= 0, 
                       1 / (1 + np.exp(-x)), 
                       np.exp(x) / (1 + np.exp(x)))
    
    def softmax(self, x):
        """Numerically stable softmax."""
        e_x = np.exp(x - np.max(x))
        return e_x / e_x.sum(axis=0)
    
    def forward(self, inputs, h_prev, c_prev):
        """
        Forward pass through entire sequence.
        
        Args:
            inputs: list of input vectors, each (d, 1)
            h_prev: initial hidden state (n, 1)
            c_prev: initial cell state (n, 1)
            
        Returns:
            outputs: list of output logits
            (h_final, c_final): final states
        """
        T = len(inputs)
        
        # Storage for backpropagation
        self.cache = {
            'inputs': inputs,
            'h': {-1: h_prev.copy()},
            'c': {-1: c_prev.copy()},
            'f_gate': {},
            'i_gate': {},
            'c_candidate': {},
            'o_gate': {},
            'concat': {}
        }
        
        outputs = []
        h_t = h_prev
        c_t = c_prev
        
        for t in range(T):
            x_t = inputs[t]
            
            # Concatenate [h_{t-1}, x_t]
            concat = np.vstack([h_t, x_t])  # (n+d, 1)
            self.cache['concat'][t] = concat
            
            # Forget gate: f_t = σ(W_f · [h_{t-1}, x_t] + b_f)
            f_t = self.sigmoid(self.W_f @ concat + self.b_f)
            self.cache['f_gate'][t] = f_t
            
            # Input gate: i_t = σ(W_i · [h_{t-1}, x_t] + b_i)
            i_t = self.sigmoid(self.W_i @ concat + self.b_i)
            self.cache['i_gate'][t] = i_t
            
            # Candidate: C̃_t = tanh(W_c · [h_{t-1}, x_t] + b_c)
            c_candidate = np.tanh(self.W_c @ concat + self.b_c)
            self.cache['c_candidate'][t] = c_candidate
            
            # Cell state update: C_t = f_t ⊙ C_{t-1} + i_t ⊙ C̃_t
            c_t = f_t * c_t + i_t * c_candidate
            self.cache['c'][t] = c_t.copy()
            
            # Output gate: o_t = σ(W_o · [h_{t-1}, x_t] + b_o)
            o_t = self.sigmoid(self.W_o @ concat + self.b_o)
            self.cache['o_gate'][t] = o_t
            
            # Hidden state: h_t = o_t ⊙ tanh(C_t)
            h_t = o_t * np.tanh(c_t)
            self.cache['h'][t] = h_t.copy()
            
            # Output: y_t = W_y · h_t + b_y
            y_t = self.W_y @ h_t + self.b_y
            outputs.append(y_t)
        
        return outputs, (h_t, c_t)
    
    def compute_loss(self, outputs, targets):
        """Cross-entropy loss."""
        loss = 0.0
        self._probs = []
        for y, t in zip(outputs, targets):
            p = self.softmax(y)
            self._probs.append(p)
            loss += -np.log(p[t, 0] + 1e-8)
        return loss
    
    def backward(self, targets):
        """
        BPTT for LSTM - backpropagates through all gates.
        
        Key insight: gradients flow through both the cell state path
        (gated by f_t) and the hidden state path (through gate computations).
        """
        T = len(targets)
        cache = self.cache
        
        # Initialize gradients
        dW_f = np.zeros_like(self.W_f)
        dW_i = np.zeros_like(self.W_i)
        dW_c = np.zeros_like(self.W_c)
        dW_o = np.zeros_like(self.W_o)
        dW_y = np.zeros_like(self.W_y)
        db_f = np.zeros_like(self.b_f)
        db_i = np.zeros_like(self.b_i)
        db_c = np.zeros_like(self.b_c)
        db_o = np.zeros_like(self.b_o)
        db_y = np.zeros_like(self.b_y)
        
        # Gradients flowing from future time steps
        dh_next = np.zeros((self.hidden_size, 1))
        dc_next = np.zeros((self.hidden_size, 1))
        
        for t in reversed(range(T)):
            # Gradient from output loss
            dy = self._probs[t].copy()
            dy[targets[t]] -= 1
            
            # Output layer gradients
            dW_y += dy @ cache['h'][t].T
            db_y += dy
            
            # Gradient into h_t (from output + from future)
            dh = self.W_y.T @ dy + dh_next
            
            # Gradient into cell state through h_t = o_t ⊙ tanh(C_t)
            tanh_c = np.tanh(cache['c'][t])
            do = dh * tanh_c  # gradient into output gate
            dc = dh * cache['o_gate'][t] * (1 - tanh_c ** 2) + dc_next
            
            # Gradient through cell state: C_t = f_t ⊙ C_{t-1} + i_t ⊙ C̃_t
            df = dc * cache['c'][t - 1]         # gradient into forget gate
            di = dc * cache['c_candidate'][t]   # gradient into input gate
            dc_candidate = dc * cache['i_gate'][t]  # gradient into candidate
            
            # Backprop through activations
            # sigmoid derivative: σ'(x) = σ(x)(1 - σ(x))
            df_raw = df * cache['f_gate'][t] * (1 - cache['f_gate'][t])
            di_raw = di * cache['i_gate'][t] * (1 - cache['i_gate'][t])
            do_raw = do * cache['o_gate'][t] * (1 - cache['o_gate'][t])
            # tanh derivative: tanh'(x) = 1 - tanh^2(x)
            dc_raw = dc_candidate * (1 - cache['c_candidate'][t] ** 2)
            
            # Parameter gradients
            concat = cache['concat'][t]
            dW_f += df_raw @ concat.T
            dW_i += di_raw @ concat.T
            dW_c += dc_raw @ concat.T
            dW_o += do_raw @ concat.T
            db_f += df_raw
            db_i += di_raw
            db_c += dc_raw
            db_o += do_raw
            
            # Gradient into concatenated input [h_{t-1}, x_t]
            d_concat = (self.W_f.T @ df_raw + self.W_i.T @ di_raw + 
                       self.W_c.T @ dc_raw + self.W_o.T @ do_raw)
            
            # Split gradient for h_{t-1} (propagates to previous time step)
            dh_next = d_concat[:self.hidden_size]
            
            # Cell state gradient propagates through forget gate
            dc_next = dc * cache['f_gate'][t]
        
        # Gradient clipping
        grads = [dW_f, dW_i, dW_c, dW_o, dW_y, db_f, db_i, db_c, db_o, db_y]
        for g in grads:
            np.clip(g, -5, 5, out=g)
        
        return {
            'dW_f': dW_f, 'dW_i': dW_i, 'dW_c': dW_c, 'dW_o': dW_o, 'dW_y': dW_y,
            'db_f': db_f, 'db_i': db_i, 'db_c': db_c, 'db_o': db_o, 'db_y': db_y
        }
    
    def update(self, grads):
        """SGD parameter update."""
        self.W_f -= self.lr * grads['dW_f']
        self.W_i -= self.lr * grads['dW_i']
        self.W_c -= self.lr * grads['dW_c']
        self.W_o -= self.lr * grads['dW_o']
        self.W_y -= self.lr * grads['dW_y']
        self.b_f -= self.lr * grads['db_f']
        self.b_i -= self.lr * grads['db_i']
        self.b_c -= self.lr * grads['db_c']
        self.b_o -= self.lr * grads['db_o']
        self.b_y -= self.lr * grads['db_y']


# --- Demo: Character-level language model with LSTM ---
print("="*60)
print("LSTM FROM SCRATCH - Character-level Language Model")
print("="*60)

# Same corpus as RNN for fair comparison
text = "hello world hello neural network"
chars = sorted(set(text))
char_to_idx = {c: i for i, c in enumerate(chars)}
idx_to_char = {i: c for i, c in enumerate(chars)}
vocab_size = len(chars)

hidden_size = 64
seq_length = 10

print(f"\nCorpus: '{text}'")
print(f"Vocabulary size: {vocab_size}, Hidden size: {hidden_size}")
print(f"Parameters: ~{4 * hidden_size * (hidden_size + vocab_size) + hidden_size * vocab_size:,} (4 gates + output)")

# Initialize LSTM
lstm = LSTMFromScratch(input_size=vocab_size, hidden_size=hidden_size, output_size=vocab_size, lr=0.01)

# Training
losses = []
for epoch in range(200):
    h_prev = np.zeros((hidden_size, 1))
    c_prev = np.zeros((hidden_size, 1))
    epoch_loss = 0
    
    for i in range(0, len(text) - seq_length, seq_length):
        inputs = []
        targets = []
        for j in range(seq_length):
            x = np.zeros((vocab_size, 1))
            x[char_to_idx[text[i + j]]] = 1
            inputs.append(x)
            targets.append(char_to_idx[text[i + j + 1]] if i + j + 1 < len(text) else char_to_idx[text[0]])
        
        # Forward
        outputs, (h_prev, c_prev) = lstm.forward(inputs, h_prev, c_prev)
        loss = lstm.compute_loss(outputs, targets)
        epoch_loss += loss
        
        # Backward & Update
        grads = lstm.backward(targets)
        lstm.update(grads)
    
    losses.append(epoch_loss)
    if epoch % 50 == 0:
        print(f"Epoch {epoch:3d} | Loss: {epoch_loss:.4f}")

print(f"\nFinal Loss: {losses[-1]:.4f} (started at {losses[0]:.4f})")
print(f"Loss reduction: {((losses[0] - losses[-1]) / losses[0] * 100):.1f}%")

LSTM FROM SCRATCH - Character-level Language Model

Corpus: 'hello world hello neural network'
Vocabulary size: 13, Hidden size: 64
Parameters: ~20,544 (4 gates + output)
Epoch   0 | Loss: 76.3575
Epoch  50 | Loss: 63.4498
Epoch 100 | Loss: 48.3378
Epoch 150 | Loss: 25.3079

Final Loss: 11.3527 (started at 76.3575)
Loss reduction: 85.1%


In [0]:
import torch
import torch.nn as nn
import torch.optim as optim

class LSTMPyTorch(nn.Module):
    """
    LSTM implemented using PyTorch's nn.Module.
    
    PyTorch's nn.LSTM internally computes all four gates in a single
    fused operation for efficiency:
        gates = W_ih @ x_t + b_ih + W_hh @ h_{t-1} + b_hh
        i, f, g, o = gates.chunk(4)  # split into 4 equal parts
        c_t = sigmoid(f) * c_{t-1} + sigmoid(i) * tanh(g)
        h_t = sigmoid(o) * tanh(c_t)
    """
    
    def __init__(self, input_size, hidden_size, output_size, num_layers=1):
        super().__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        
        # PyTorch's fused LSTM implementation
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True
        )
        
        # Output projection
        self.fc = nn.Linear(hidden_size, output_size)
    
    def forward(self, x, states=None):
        """
        Args:
            x: (batch_size, seq_len, input_size)
            states: tuple of (h_0, c_0), each (num_layers, batch, hidden_size)
        Returns:
            output: (batch_size, seq_len, output_size)
            (h_n, c_n): final hidden and cell states
        """
        if states is None:
            h_0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size, device=x.device)
            c_0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size, device=x.device)
            states = (h_0, c_0)
        
        # lstm_out: all hidden states, (h_n, c_n): final states
        lstm_out, (h_n, c_n) = self.lstm(x, states)
        
        # Project to output space
        output = self.fc(lstm_out)
        return output, (h_n, c_n)


# --- Demo: Character-level task with PyTorch LSTM ---
print("="*60)
print("LSTM USING PYTORCH - Character-level Language Model")
print("="*60)

# Same data preparation as PyTorch RNN
text = "hello world hello neural network"
chars = sorted(set(text))
char_to_idx = {c: i for i, c in enumerate(chars)}
vocab_size = len(chars)
seq_length = 10
hidden_size = 64

encoded = [char_to_idx[c] for c in text]

X_seqs, Y_seqs = [], []
for i in range(0, len(encoded) - seq_length):
    X_seqs.append(encoded[i:i+seq_length])
    Y_seqs.append(encoded[i+1:i+seq_length+1])

X_tensor = torch.zeros(len(X_seqs), seq_length, vocab_size)
for i, seq in enumerate(X_seqs):
    for j, idx in enumerate(seq):
        X_tensor[i, j, idx] = 1.0
Y_tensor = torch.LongTensor(Y_seqs)

# Initialize model
lstm_model = LSTMPyTorch(input_size=vocab_size, hidden_size=hidden_size, output_size=vocab_size)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(lstm_model.parameters(), lr=0.01)

# Parameter comparison with RNN
total_params = sum(p.numel() for p in lstm_model.parameters())
lstm_params = sum(p.numel() for p in lstm_model.lstm.parameters())
print(f"\nTotal parameters: {total_params:,}")
print(f"  LSTM layer: {lstm_params:,} (≈4x RNN due to 4 gates)")
print(f"  FC layer: {sum(p.numel() for p in lstm_model.fc.parameters()):,}")

# Training
losses = []
for epoch in range(200):
    optimizer.zero_grad()
    output, _ = lstm_model(X_tensor)
    loss = criterion(output.view(-1, vocab_size), Y_tensor.view(-1))
    loss.backward()
    torch.nn.utils.clip_grad_norm_(lstm_model.parameters(), max_norm=5.0)
    optimizer.step()
    
    losses.append(loss.item())
    if epoch % 50 == 0:
        print(f"Epoch {epoch:3d} | Loss: {loss.item():.4f}")

print(f"\nFinal Loss: {losses[-1]:.4f}")

# Text generation
lstm_model.eval()
with torch.no_grad():
    seed = "hell"
    generated = seed
    states = None
    
    for char in seed:
        x = torch.zeros(1, 1, vocab_size)
        x[0, 0, char_to_idx[char]] = 1.0
        _, states = lstm_model(x, states)
    
    x = torch.zeros(1, 1, vocab_size)
    x[0, 0, char_to_idx[seed[-1]]] = 1.0
    
    for _ in range(20):
        out, states = lstm_model(x, states)
        probs = torch.softmax(out[0, 0], dim=0)
        idx = torch.multinomial(probs, 1).item()
        generated += chars[idx]
        x = torch.zeros(1, 1, vocab_size)
        x[0, 0, idx] = 1.0
    
    print(f"\nGenerated text (seed='{seed}'): '{generated}'")

LSTM USING PYTORCH - Character-level Language Model

Total parameters: 21,069
  LSTM layer: 20,224 (≈4x RNN due to 4 gates)
  FC layer: 845
Epoch   0 | Loss: 2.5747
Epoch  50 | Loss: 0.1432
Epoch 100 | Loss: 0.1023
Epoch 150 | Loss: 0.0979

Final Loss: 0.0964

Generated text (seed='hell'): 'hell neural networkkwhhl'


---
# Part 3: Time Complexity & Comparison Summary

## 3.1 Computational Complexity

Let: $$T$$ = sequence length, $$n$$ = hidden size, $$d$$ = input size, $$o$$ = output size, $$L$$ = number of layers

| Metric | Vanilla RNN | LSTM |
|--------|------------|------|
| **FLOPs per step** | $$O(n^2 + nd + no)$$ | $$O(4(n^2 + nd) + no)$$ |
| **FLOPs full sequence** | $$O(T(n^2 + nd))$$ | $$O(4T(n^2 + nd))$$ |
| **Parameters** | $$n^2 + nd + no + on + \text{biases}$$ | $$4(n^2 + nd) + no + on + \text{biases}$$ |
| **Parameter count (approx.)** | $$\approx n^2 + nd$$ | $$\approx 4(n^2 + nd)$$ |
| **Space (activations)** | $$O(Tn)$$ | $$O(6Tn)$$ — store $$h, C, f, i, \tilde{C}, o$$ |
| **Parallelizable across time?** | No | No |
| **Multi-layer** | $$O(TLn^2)$$ | $$O(4TLn^2)$$ |

## 3.2 Concrete Parameter Counts (our demo)

With $$d = 13$$ (vocab), $$n = 64$$ (hidden), $$o = 13$$ (output):

| Model | Formula | Count |
|-------|---------|-------|
| RNN | $$n \times d + n \times n + n + o \times n + o$$ | $$64 \times 13 + 64 \times 64 + 64 + 13 \times 64 + 13 = 5,581$$ |
| LSTM | $$4 \times n \times (n + d) + 4 \times n + o \times n + o$$ | $$4 \times 64 \times 77 + 256 + 13 \times 64 + 13 = 20,557$$ |
| **LSTM / RNN ratio** | | **\~3.7x** more parameters |

## 3.3 Training Considerations

| Aspect | RNN | LSTM |
|--------|-----|------|
| Gradient flow | Degrades exponentially with $$T$$ | Stable via cell state highway |
| Effective memory | \~5-10 steps in practice | 100+ steps possible |
| Training speed per epoch | Faster (fewer ops) | Slower (4x computation) |
| Convergence on long sequences | Often fails | Reliable |
| Gradient clipping needed? | Critical | Helpful but less critical |

## 3.4 When to Use What

| Use Case | Recommendation | Reason |
|----------|---------------|--------|
| Short sequences ($$T < 20$$) | RNN | Simpler, faster, sufficient |
| Long sequences ($$T > 50$$) | LSTM (or GRU) | RNN cannot learn long dependencies |
| Resource-constrained | GRU | 3 gates (vs LSTM's 4), similar performance |
| State-of-the-art NLP/speech | Transformer | $$O(T^2d)$$ but fully parallelizable |
| Real-time streaming | LSTM / GRU | Low latency per step |

In [0]:
import time
import torch
import torch.nn as nn

def benchmark_model(model_class, model_kwargs, input_tensor, targets, n_epochs=100):
    """Benchmark training time for a model."""
    model = model_class(**model_kwargs)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.01)
    
    start = time.time()
    for _ in range(n_epochs):
        optimizer.zero_grad()
        output, _ = model(input_tensor)
        loss = criterion(output.view(-1, model_kwargs['output_size']), targets.view(-1))
        loss.backward()
        optimizer.step()
    elapsed = time.time() - start
    
    return elapsed, loss.item()

# Benchmark setup
print("="*60)
print("TIMING BENCHMARK: RNN vs LSTM (PyTorch)")
print("="*60)

seq_lengths = [10, 25, 50]
hidden_sizes = [32, 64, 128]

print(f"\n{'Model':<8} {'Seq Len':<10} {'Hidden':<8} {'Time (s)':<12} {'Final Loss':<12} {'Params':<10}")
print("-" * 60)

for seq_len in seq_lengths:
    # Generate data for this sequence length
    n_samples = 50
    X = torch.randn(n_samples, seq_len, vocab_size)
    Y = torch.randint(0, vocab_size, (n_samples, seq_len))
    
    for h_size in [64]:
        # RNN
        rnn_time, rnn_loss = benchmark_model(
            RNNPyTorch, 
            {'input_size': vocab_size, 'hidden_size': h_size, 'output_size': vocab_size},
            X, Y, n_epochs=100
        )
        rnn_params = sum(p.numel() for p in RNNPyTorch(vocab_size, h_size, vocab_size).parameters())
        
        # LSTM
        lstm_time, lstm_loss = benchmark_model(
            LSTMPyTorch,
            {'input_size': vocab_size, 'hidden_size': h_size, 'output_size': vocab_size},
            X, Y, n_epochs=100
        )
        lstm_params = sum(p.numel() for p in LSTMPyTorch(vocab_size, h_size, vocab_size).parameters())
        
        print(f"{'RNN':<8} {seq_len:<10} {h_size:<8} {rnn_time:<12.4f} {rnn_loss:<12.4f} {rnn_params:<10,}")
        print(f"{'LSTM':<8} {seq_len:<10} {h_size:<8} {lstm_time:<12.4f} {lstm_loss:<12.4f} {lstm_params:<10,}")
        print(f"{'Ratio':<8} {'':10} {'':8} {lstm_time/rnn_time:<12.2f}x")
        print()

print("\n" + "="*60)
print("KEY TAKEAWAYS:")
print("="*60)
print("""
1. LSTM is ~2-4x slower per epoch than RNN (4 gates vs 1 update)
2. LSTM has ~4x more parameters than RNN for same hidden size
3. Both scale linearly with sequence length T (sequential dependency)
4. LSTM achieves better loss on longer sequences (gradient highway)
5. Neither can be parallelized across time steps (unlike Transformers)
""")

TIMING BENCHMARK: RNN vs LSTM (PyTorch)

Model    Seq Len    Hidden   Time (s)     Final Loss   Params    
------------------------------------------------------------
RNN      10         64       0.1911       0.0236       5,901     
LSTM     10         64       0.1823       0.0199       21,069    
Ratio                        0.95        x

RNN      25         64       0.3486       0.3301       5,901     
LSTM     25         64       0.4296       0.0826       21,069    
Ratio                        1.23        x

RNN      50         64       0.6393       1.1695       5,901     
LSTM     50         64       0.6505       0.2617       21,069    
Ratio                        1.02        x


KEY TAKEAWAYS:

1. LSTM is ~2-4x slower per epoch than RNN (4 gates vs 1 update)
2. LSTM has ~4x more parameters than RNN for same hidden size
3. Both scale linearly with sequence length T (sequential dependency)
4. LSTM achieves better loss on longer sequences (gradient highway)
5. Neither can be para